# Module 1.4 -- Real-Time Event Streaming & Live Semantic Layers

**Track:** Big Data Engineering for AI Systems  
**Toolchain:** Apache Kafka - Apache Flink - Debezium CDC - Vector Databases  
**Objective:** Build event-driven pipelines that continuously feed AI systems with millisecond-fresh features and live vector embeddings.

---

## Why Batch Processing Is Not Enough for AI

**Scenario:** A fraud detection model runs hourly batch inference. A fraudster makes 50 transactions in 30 minutes. The model doesn't see them until the NEXT batch run -- by then, $50,000 is gone.

**With streaming:** Every transaction hits the model in <100ms. Transaction #3 is flagged and blocked.

| | Batch | Micro-Batch (Spark) | True Streaming (Flink) |
|---|---|---|---|
| Latency | Minutes-hours | 1-10 seconds | 1-50 milliseconds |
| Best for | Reports, training data | Near-real-time features | Fraud, trading, live personalization |
| Complexity | Low | Medium | High |
| Cost | Low | Medium | Higher |

### Key Streaming Concepts for Beginners

| Term | Meaning | Analogy |
|------|---------|--------|
| **Event** | A single data record (e.g., one click) | A letter arriving |
| **Topic** | A named channel for events | A mailbox labeled 'orders' |
| **Producer** | Sends events to a topic | The person mailing letters |
| **Consumer** | Reads events from a topic | The person checking the mailbox |
| **Offset** | Position in the event stream | Page number in a book |
| **CDC** | Change Data Capture -- stream DB changes | A security camera recording every change |


## 📑 Table of Contents

* [Why Batch Processing Is Not Enough for AI](#why-batch-processing-is-not-enough-for-ai)
  * [Key Streaming Concepts for Beginners](#key-streaming-concepts-for-beginners)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [Syntax Bridge: SQL for Streaming](#syntax-bridge-sql-for-streaming)
* [1 - Apache Kafka: The Event Backbone](#1-apache-kafka-the-event-backbone)
  * [What is Kafka?](#what-is-kafka)
  * [Diskless Kafka (2026)](#diskless-kafka-2026)
  * [Kafka Partitions & Consumer Groups (Critical Concept)](#kafka-partitions-consumer-groups-critical-concept)
* [2 - Apache Flink: Stateful Stream Processing](#2-apache-flink-stateful-stream-processing)
  * [Why Flink Instead of Spark Streaming?](#why-flink-instead-of-spark-streaming)
  * [Flink SQL: Stream Processing Without Java](#flink-sql-stream-processing-without-java)
* [3 - CDC with Debezium: Streaming Database Changes](#3-cdc-with-debezium-streaming-database-changes)
  * [What is CDC (Change Data Capture)?](#what-is-cdc-change-data-capture)
  * [When to Use CDC vs Direct Queries](#when-to-use-cdc-vs-direct-queries)
* [4 - Live Semantic Layer: Real-Time Vector Embeddings](#4-live-semantic-layer-real-time-vector-embeddings)
  * [The Architecture](#the-architecture)
* [Production Reality Check](#production-reality-check)
  * [Real Kafka Setup Complexity](#real-kafka-setup-complexity)
* [Knowledge Check](#knowledge-check)
  * [Question 1: Kafka vs Message Queue](#question-1-kafka-vs-message-queue)
  * [Question 2: Watermarks](#question-2-watermarks)
  * [Question 3: CDC vs Polling](#question-3-cdc-vs-polling)
  * [Question 4: Exactly-Once Semantics](#question-4-exactly-once-semantics)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Kafka Partition Assignment](#exercise-1-kafka-partition-assignment)
  * [Exercise 2: Window Types](#exercise-2-window-types)
  * [Exercise 3: CDC Event Processing](#exercise-3-cdc-event-processing)
* [Summary](#summary)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Batch processing (once a day) is too slow for modern AI (fraud detection, real-time recommendations). Seniors build event-driven architectures where data flows continuously like water.

**Mental Model:** Kafka is a massive, indestructible conveyor belt of events. Flink is the robotic arm reading items off the belt, transforming them instantly, and placing them on a new belt.

**Common Junior Pitfall:** Trying to build a "streaming" system using cron jobs that run every 1 minute. Real streaming requires persistent open connections and state management.

---


---

## Syntax Bridge: SQL for Streaming

If you haven't written SQL before, here's a 60-second crash course (covered in detail in Notebook 00):

```sql
SELECT column FROM table WHERE condition;              -- Filter rows
SELECT col, COUNT(*) FROM table GROUP BY col;           -- Count per group
SELECT *, SUM(x) OVER (PARTITION BY y) FROM table;      -- Running total per group
```

**Flink SQL = regular SQL, but the table is a live stream**. Instead of querying stored data, you query data AS IT FLOWS through. The `TUMBLE()` function creates time windows (e.g., every 5 minutes).


In [ ]:
# Cell 1 -- Install
!pip install -q confluent-kafka faust-streaming pandas numpy


---
## 1 - Apache Kafka: The Event Backbone

### What is Kafka?

Kafka is a **distributed event log**. Every event ever written is stored (until retention expires). Unlike a queue (where reading removes the message), Kafka lets multiple consumers read the same events independently.

```
Producer A --\                  /--> Consumer 1 (ML inference)
Producer B ----> [Kafka Topic] ----> Consumer 2 (Dashboard)
Producer C --/                  \--> Consumer 3 (Archival)
```

### Diskless Kafka (2026)

Traditional Kafka stores everything on local SSDs (expensive, hard to scale). **Diskless Kafka** stores historical data in cloud object storage (S3) while keeping recent data in memory.

| | Traditional Kafka | Diskless Kafka |
|---|---|---|
| Storage | Local SSDs | S3/GCS (10x cheaper) |
| Retention | Days-weeks (disk limited) | Months-years (unlimited) |
| Scaling | Add disks (slow) | Add compute (fast) |
| Hot data latency | <5ms | <5ms (in-memory cache) |
| Cold data latency | Same | 50-200ms (S3 fetch) |


In [ ]:
# Cell 2 -- Kafka producer and consumer simulation
import json, time, random
from datetime import datetime

class KafkaSimulator:
    def __init__(self):
        self.topics = {}

    def produce(self, topic, key, value):
        if topic not in self.topics: self.topics[topic] = []
        event = {'key': key, 'value': value, 'timestamp': datetime.utcnow().isoformat(), 'offset': len(self.topics[topic])}
        self.topics[topic].append(event)
        return event['offset']

    def consume(self, topic, from_offset=0, limit=5):
        return self.topics.get(topic, [])[from_offset:from_offset+limit]

kafka = KafkaSimulator()

# Simulate e-commerce events
products = ['laptop', 'phone', 'headphones', 'tablet', 'watch']
for i in range(20):
    event = {'user_id': f'user_{random.randint(1,100)}', 'product': random.choice(products),
             'action': random.choice(['view', 'click', 'purchase']), 'price': round(random.uniform(10, 2000), 2)}
    kafka.produce('user-events', event['user_id'], event)

print(f'Produced {len(kafka.topics["user-events"])} events to topic: user-events')
print(f'\nSample events (offset 0-4):')
for e in kafka.consume('user-events', 0, 5):
    print(f'  [{e["offset"]}] {e["value"]["user_id"]}: {e["value"]["action"]} {e["value"]["product"]} ${e["value"]["price"]}')


---
### Kafka Partitions & Consumer Groups (Critical Concept)

A Kafka topic is split into **partitions**. This is how Kafka scales:

```
Topic: user-events (3 partitions)
  Partition 0: [event0, event3, event6, ...]  -> Consumer A
  Partition 1: [event1, event4, event7, ...]  -> Consumer B
  Partition 2: [event2, event5, event8, ...]  -> Consumer C
```

**Key Rules:**
- Events with the same key ALWAYS go to the same partition (ordering guaranteed per key)
- Each partition is consumed by exactly ONE consumer in a consumer group
- Adding more consumers (up to # partitions) = parallel processing
- Adding more consumers than partitions = idle consumers (waste!)

| Partitions | Consumers | Result |
|:---:|:---:|---|
| 3 | 1 | 1 consumer reads all 3 partitions (slow) |
| 3 | 3 | 1 consumer per partition (optimal) |
| 3 | 6 | 3 consumers idle (wasted resources) |


In [ ]:
# Windowed aggregation simulation
# This simulates what Flink's TUMBLE() window does
from collections import defaultdict
from datetime import datetime, timedelta
import random

# Generate timestamped events
events = []
base = datetime(2026, 3, 15, 10, 0, 0)
for i in range(50):
    events.append({
        'user_id': f'user_{random.randint(1,5)}',
        'action': random.choice(['view', 'click', 'purchase']),
        'amount': round(random.uniform(5, 200), 2),
        'timestamp': base + timedelta(minutes=random.randint(0, 29))
    })

# 10-minute tumbling window aggregation
WINDOW_SIZE = timedelta(minutes=10)
windows = defaultdict(lambda: {'event_count': 0, 'total_amount': 0, 'purchases': 0})

for e in events:
    # Calculate which window this event belongs to
    minutes_since_start = (e['timestamp'] - base).total_seconds() / 60
    window_num = int(minutes_since_start // 10)
    window_start = base + timedelta(minutes=window_num * 10)
    key = (e['user_id'], window_start.strftime('%H:%M'))
    
    windows[key]['event_count'] += 1
    windows[key]['total_amount'] += e['amount']
    if e['action'] == 'purchase':
        windows[key]['purchases'] += 1

print('10-Minute Tumbling Window Aggregation:')
print(f'{"User":<10} {"Window":<10} {"Events":<8} {"Purchases":<10} {"Amount"}')
print('-' * 50)
for (user, window), stats in sorted(windows.items()):
    print(f'{user:<10} {window:<10} {stats["event_count"]:<8} {stats["purchases"]:<10} ${stats["total_amount"]:.2f}')
print('\nThis is exactly what Flink SQL TUMBLE() does -- but on a live stream!')


---
## 2 - Apache Flink: Stateful Stream Processing

### Why Flink Instead of Spark Streaming?

| Feature | Flink | Spark Streaming |
|---------|-------|----------------|
| True event-at-a-time | Yes | No (micro-batch) |
| Event time processing | Native | Bolt-on |
| Exactly-once semantics | Built-in | Requires config |
| Savepoints (pause/resume) | Yes | No |
| SQL interface | Flink SQL (mature) | Structured Streaming |

### Flink SQL: Stream Processing Without Java

You can write streaming logic in SQL(!). Flink translates SQL to a streaming execution plan.


In [ ]:
# Cell 3 -- Flink SQL definition (conceptual)
flink_sql = '''
-- Create a Kafka source table (connects Flink to Kafka topic)
CREATE TABLE user_events (
    user_id    STRING,
    product    STRING,
    action     STRING,
    price      DECIMAL(10, 2),
    event_time TIMESTAMP(3),
    WATERMARK FOR event_time AS event_time - INTERVAL '5' SECOND
) WITH (
    'connector' = 'kafka',
    'topic' = 'user-events',
    'properties.bootstrap.servers' = 'kafka:9092',
    'format' = 'json'
);

-- Real-time aggregation: 5-minute tumbling windows
CREATE TABLE live_features AS
SELECT
    user_id,
    TUMBLE_START(event_time, INTERVAL '5' MINUTE) AS window_start,
    COUNT(*) AS event_count,
    COUNT(DISTINCT product) AS unique_products,
    SUM(CASE WHEN action = 'purchase' THEN price ELSE 0 END) AS total_spent,
    SUM(CASE WHEN action = 'purchase' THEN 1 ELSE 0 END) AS purchase_count
FROM user_events
GROUP BY user_id, TUMBLE(event_time, INTERVAL '5' MINUTE);
'''
print('Flink SQL -- Real-Time Feature Engineering:')
print(flink_sql)
print('This SQL runs CONTINUOUSLY, updating live_features every 5 minutes.')


---
## 3 - CDC with Debezium: Streaming Database Changes

### What is CDC (Change Data Capture)?

CDC captures every INSERT, UPDATE, DELETE from your database and streams it as events. Instead of querying the database periodically (expensive), you react to changes as they happen.

```
PostgreSQL (source)  -->  Debezium  -->  Kafka  -->  Flink  -->  Feature Store
    |                      |                                        |
  User updates        Captures the        Processes              ML model reads
  their profile       row change           in real-time          fresh features
```

### When to Use CDC vs Direct Queries

| | Direct DB Queries | CDC Streaming |
|---|---|---|
| Latency | Query interval (1min-1hr) | Milliseconds |
| DB load | High (repeated full scans) | Near-zero |
| Data freshness | Stale by query interval | Always current |
| Complexity | Low (just SQL) | Medium (Debezium + Kafka) |


In [ ]:
# Cell 4 -- Simulated CDC event processing
import json

cdc_events = [
    {'op': 'c', 'table': 'users', 'before': None, 'after': {'id': 1, 'name': 'Alice', 'credit_score': 720}},
    {'op': 'u', 'table': 'users', 'before': {'id': 1, 'name': 'Alice', 'credit_score': 720}, 'after': {'id': 1, 'name': 'Alice', 'credit_score': 745}},
    {'op': 'c', 'table': 'orders', 'before': None, 'after': {'id': 101, 'user_id': 1, 'amount': 250.00, 'product': 'laptop_stand'}},
    {'op': 'd', 'table': 'orders', 'before': {'id': 99, 'user_id': 1, 'amount': 15.00, 'product': 'cable'}, 'after': None},
]

OP_NAMES = {'c': 'INSERT', 'u': 'UPDATE', 'd': 'DELETE', 'r': 'SNAPSHOT'}

print('CDC Event Stream (from Debezium):')
for e in cdc_events:
    op = OP_NAMES[e['op']]
    if e['op'] == 'c': print(f'  + {op} {e["table"]}: {e["after"]}')
    elif e['op'] == 'u': print(f'  ~ {op} {e["table"]}: credit_score {e["before"]["credit_score"]} -> {e["after"]["credit_score"]}')
    elif e['op'] == 'd': print(f'  - {op} {e["table"]}: removed order #{e["before"]["id"]}')
print(f'\nResult: ML feature store updated in real-time (no batch delay)')


---
## 4 - Live Semantic Layer: Real-Time Vector Embeddings

### The Architecture

```
Kafka (text events) --> Flink (generate embeddings) --> Vector DB --> RAG System
```

By generating embeddings in the STREAMING pipeline (not batch), your RAG system always has the freshest context. No more hallucinating about yesterday's data.


In [ ]:
# Cell 5 -- Simulate real-time embedding generation
import numpy as np

incoming_docs = [
    'New product launched: AI-powered code review tool',
    'Q4 earnings exceeded expectations by 15%',
    'CEO announced expansion into European markets',
    'Security patch CVE-2026-1234 deployed to all servers',
]

print('Live Semantic Layer -- Real-Time Embedding Pipeline:')
for doc in incoming_docs:
    # In production: embedding = model.encode(doc)
    embedding = np.random.randn(384)
    norm = np.linalg.norm(embedding)
    print(f'  Event: "{doc[:50]}..."')
    print(f'    -> Embedding: [{embedding[0]:.3f}, {embedding[1]:.3f}, ...] (384-dim, norm={norm:.2f})')
    print(f'    -> Pushed to vector DB (available for RAG in <100ms)')
print(f'\nTotal: {len(incoming_docs)} documents embedded and indexed in real-time')


---

## Production Reality Check

| In This Notebook | In Production |
|---|---|
| `KafkaSimulator` (Python dict) | Real Kafka cluster (3+ brokers, ZooKeeper/KRaft) |
| Simulated CDC events | Debezium connector reading PostgreSQL WAL |
| Random embeddings | Sentence-transformers or OpenAI embedding API |
| In-memory processing | Flink cluster with checkpointing to S3 |

### Real Kafka Setup Complexity

| Concern | What to Handle |
|---------|---------------|
| Serialization | Avro/Protobuf schemas in Schema Registry |
| Consumer groups | Rebalancing when consumers join/leave |
| Exactly-once | Idempotent producers + transactional consumers |
| Monitoring | Lag metrics, throughput, consumer health |
| Cost | Managed Confluent: ~$0.11/GB. Self-hosted: ~$0.05/GB + ops |


---
## Knowledge Check

### Question 1: Kafka vs Message Queue
What is the fundamental difference between Kafka and a traditional message queue (like RabbitMQ)?

<details>
<summary>Click to reveal answer</summary>

In a traditional queue, reading a message **removes** it. In Kafka, reading does NOT remove the message -- it stays in the log until retention expires. Multiple consumers can independently read the same events at their own pace. This makes Kafka ideal for **event sourcing** where many downstream systems need the same data.
</details>

### Question 2: Watermarks
In the Flink SQL example, what does `WATERMARK FOR event_time AS event_time - INTERVAL '5' SECOND` mean?

<details>
<summary>Click to reveal answer</summary>

It tells Flink to allow events to arrive **up to 5 seconds late**. If the window closes at 10:05:00, Flink will still accept events with `event_time` between 10:04:55 and 10:05:00 that arrive after 10:05:00. Events arriving more than 5 seconds late are dropped. This handles network delays and out-of-order events.
</details>

### Question 3: CDC vs Polling
A team polls their PostgreSQL database every 60 seconds to check for new orders. What problems does CDC solve?

<details>
<summary>Click to reveal answer</summary>

CDC solves: (1) **Latency** -- changes are captured in milliseconds, not 60-second intervals. (2) **Database load** -- polling requires repeated `SELECT *` queries that load the DB; CDC reads the write-ahead log (WAL) with near-zero impact. (3) **Missing deletes** -- polling misses DELETE operations unless you add `deleted_at` columns; CDC captures all operation types.
</details>

### Question 4: Exactly-Once Semantics
What does 'exactly-once' processing mean, and why is it hard in streaming?

<details>
<summary>Click to reveal answer</summary>

Exactly-once means each event is processed **exactly one time** -- no duplicates, no missed events. It's hard because: (1) Network failures can cause a consumer to crash after processing but before acknowledging; (2) Retrying the event causes duplicate processing; (3) Solutions require **idempotent producers** + **transactional consumers** + **checkpointing** -- all working together.
</details>


---
## Practical Practice

### Exercise 1: Kafka Partition Assignment
You have a Kafka topic with 6 partitions and a consumer group with 4 consumers. How are partitions assigned? What happens if consumer #3 crashes?

### Exercise 2: Window Types
Explain the difference between a **tumbling window** (5 min) and a **sliding window** (5 min window, 1 min slide). When would you use each?

### Exercise 3: CDC Event Processing
Given the CDC event below, write Python code to update a dictionary (simulating a feature store) with the changed values:
```python
event = {'op': 'u', 'before': {'id': 1, 'score': 720}, 'after': {'id': 1, 'score': 745}}
```


In [ ]:
# ===== EXERCISE SOLUTIONS =====

print('Ex 1: Partition Assignment')
print('  6 partitions, 4 consumers:')
print('  Consumer 1: partitions 0, 1')
print('  Consumer 2: partitions 2, 3')
print('  Consumer 3: partitions 4')
print('  Consumer 4: partition 5')
print('  If #3 crashes: rebalance -> remaining 3 consumers split 6 partitions (2 each)')

print('\nEx 2: Window Types')
print('  Tumbling (5min): [0-5], [5-10], [10-15] -- non-overlapping, each event in exactly 1 window')
print('  Sliding (5min/1min): [0-5], [1-6], [2-7] -- overlapping, each event appears in 5 windows')
print('  Use tumbling for: reports, billing, aggregates')
print('  Use sliding for: moving averages, anomaly detection, trend analysis')

print('\nEx 3: CDC Feature Update')
feature_store = {1: {'score': 720}}
event = {'op': 'u', 'before': {'id': 1, 'score': 720}, 'after': {'id': 1, 'score': 745}}

if event['op'] in ('c', 'u'):  # INSERT or UPDATE
    entity_id = event['after']['id']
    feature_store[entity_id] = {k: v for k, v in event['after'].items() if k != 'id'}
elif event['op'] == 'd':  # DELETE
    entity_id = event['before']['id']
    feature_store.pop(entity_id, None)

print(f'  Before: {{1: {{"score": 720}}}}')
print(f'  After:  {feature_store}')


---
## Summary

| Concept | What You Learned |
|---------|------------------|
| Kafka | Distributed event log, multiple consumers |
| Diskless Kafka | Cloud storage backend, infinite retention |
| Flink SQL | Stream processing in SQL |
| CDC (Debezium) | Stream database changes, zero DB load |
| Live embeddings | Real-time vectors for RAG freshness |

**Next →** `11_feature_engineering.ipynb` — Layer 2 — Feature Engineering: Real-time Materialization & Point-in-Time Joins